# Compute Pareto front using Optuna with GMM sampling

In [1]:
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import plotly.express as px
import plotly.graph_objects as go

In [2]:
import warnings
warnings.simplefilter('ignore')

## Load the models trained

In [3]:
# load the models trained

with open('./data_common/best_RF_E.pickle', mode='rb') as f:
    rf_E = pickle.load(f)

In [4]:
with open('./data_common/best_RF_density.pickle', mode='rb') as f:
    rf_density = pickle.load(f)

In [5]:
with open('./data_common/best_RF_Tg.pickle', mode='rb') as f:
    rf_Tg = pickle.load(f)

## Load sample data created with GMM

In [6]:
df_GMM = pd.read_csv('./data_common/generated_samples_GMM.csv')

print(df_GMM.shape)
df_GMM.head()

(10002, 7)


,SiO2,B2O3,Al2O3,MgO,CaO,Na2O,K2O
0,63.272577,0.728542,2.505179,7.258545,0.249837,24.872982,1.112336
1,69.266394,4.918933,13.060380,2.744375,0.000000,10.009918,0.000000
2,72.692913,5.799882,16.096459,3.931258,0.122058,0.055953,1.301477
3,68.985202,5.180258,7.119872,0.000000,0.083171,18.138966,0.492530
4,69.831482,2.080346,10.614635,10.245749,0.254270,5.662475,1.311042


## Search in GMM (less extrapolation) area

In [7]:
# define objective functions

def objective(trial):

    idx = trial.suggest_int('IDX', 0, len(df_GMM)-1)
    composition = df_GMM.iloc[idx]
    X_input = np.array(composition).reshape(1, -1)
    
    e_pred = rf_E.predict(X_input)[0]
    density_pred = rf_density.predict(X_input)[0]
    tg_pred = rf_Tg.predict(X_input)[0]
    
    return e_pred, density_pred, tg_pred

In [8]:
# determine optimazion conditions : E(maximize), density(maximize), Tg(minimize)

study = optuna.create_study(directions=["maximize", "maximize", "minimize"])

[I 2026-09-26 09:56:02,043] A new study created in memory with name: no-name-1cbf5a44-128e-4a26-b0db-76731896ab95


In [9]:
# profibit showing log

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [10]:
# optimization process

study.optimize(objective, n_trials=100)

In [11]:
# print results

pareto_front = study.best_trials 

print(f"nunber of Pareto fronts = {len(pareto_front)}")

nunber of Pareto fronts = 15


In [12]:
# make table of the results

df_pareto_opt_comp = pd.DataFrame()

for trial in pareto_front: # study.best_trials と同じ
    idx = trial.number
    trial_pareto_opt_comp = np.array(df_GMM.iloc[idx, :]).reshape(1, -1)
    trial_values = np.array(trial.values).reshape(1, -1)
    
    trial_comp_values = np.concatenate([trial_pareto_opt_comp, trial_values], axis=1)
    df_trial_comp_values = pd.DataFrame(trial_comp_values)
    df_pareto_opt_comp = pd.concat([df_pareto_opt_comp, df_trial_comp_values], axis=0)

df_pareto_opt_comp.columns = ["SiO2", "B2O3", "Al2O3", "MgO", "CaO", "Na2O", "K2O", "E", "density", "Tg"]
print(df_pareto_opt_comp.shape)
df_pareto_opt_comp.head()

(15, 10)


,SiO2,B2O3,Al2O3,MgO,CaO,Na2O,K2O,E,density,Tg
0,52.405312,0.000000,12.244791,6.330081,0.031753,27.778936,1.209129,81.391868,2.519302,592.238435
0,61.696581,0.000000,7.674022,14.982509,0.291458,14.492013,0.863418,66.158756,2.455371,511.778912
0,60.258216,2.465599,6.678544,5.099264,0.138546,25.359831,0.000000,88.017854,2.652013,606.344388
0,59.363972,4.980362,15.325920,0.000000,0.000000,20.329746,0.000000,80.335543,2.640053,570.438776
0,69.738462,0.373190,16.061995,2.445041,0.079954,10.587069,0.714289,65.006039,2.464188,531.327806


In [13]:
# visualization

all_values = np.array([t.values for t in study.trials])           # targets from all study
pareto_values = np.array([t.values for t in study.best_trials])   # targets optimized

df_all_values = pd.DataFrame(all_values, columns = ['E', 'density', 'Tg'])        # targets from all study
df_pareto_values = pd.DataFrame(pareto_values, columns = ['E', 'density', 'Tg'])  # targets optimized
df_all_values['group'] = 'All'                                                    # add groups
df_pareto_values['group'] = 'Pareto'
df = pd.concat([df_all_values, df_pareto_values])

fig = px.scatter_3d(df, x='E', y='density', z='Tg', color='group', width=800, height=600)   # 3d plot
fig.update_layout(margin = dict(l=0, r=0, b=20, t=20))       # margins to depict

fig.update_traces(marker = dict(size = 3))
fig.show()

## Save the final results for comparison

In [14]:
# save the dataframe

df_pareto_opt_comp.to_csv('./data_common/df_pareto_opt_comp_gmm.csv', index = False)